Generates an MLP regressor & tests it. 

Import & set filepaths

In [2]:
from pathlib import Path

ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR.joinpath('data')
ONNX_MODEL_DIR = ROOT_DIR.joinpath('onnx_models')

from parquet_to_pd import parquetToDf
from csv_to_pd import csvToDf
from data_split import DataSplit
from data_ingestion import setup_split

import matplotlib.pyplot as plt
from sklearn.neural_network._multilayer_perceptron import MLPRegressor


Data ingestion & setting up split

In [3]:
splt = setup_split()
print("Length of train, test, and validation sets:", len(splt.train), len(splt.test), len(splt.val))

Length of train, test, and validation sets: 292488 62676 62677


Set up param grid to search MLP parameters

In [9]:
grid:list[tuple[int,int]] = [(a,b) for a in range(80, 160, 20) for b in range (20, 80, 20)]
param_grid = [
  {'hidden_layer_sizes': grid, "n_iter_no_change": [10]}
]

Perform MLP grid search. This cell will hold up your kernel for a while!

In [ ]:
from sklearn_optimiser import SklearnOptimiser
mlp_clr = MLPRegressor()
optimizer = SklearnOptimiser(splt, mlp_clr, param_grid)
optimizer.optimize("grid search", n_jobs=4)

C:\Users\Fourt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\Fourt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\Fourt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\Fourt\A

In [1]:
print(optimizer.getOptimalParameters())

NameError: name 'optimizer' is not defined

Convert sklearn model to onnx for porting & save model to file

In [ ]:
from skl2onnx import to_onnx
import onnx

onx = to_onnx(optimizer.getOptimalClassifier(), splt.train.to_numpy(flatten=True))
onnx.save(onx, ONNX_MODEL_DIR.joinpath("mlp_model1.onnx"))